In [ ]:
# import sys, os
# # BASE = r"G:\Meu Drive\Doutorado\bibliotecas"
# BASE = r"D:\bibliotecas"
# if BASE not in sys.path:
#     sys.path.insert(0, BASE)

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import glob
from tqdm import tqdm
from IPython.display import display
dest = os.getcwd()  # use r'' no Windows
os.makedirs(dest, exist_ok=True)  # cria recursivamente se não existir
os.chdir(dest)

# Exibir todas as colunas
pd.set_option('display.max_columns', None)

# Exibir todas as linhas
pd.set_option('display.max_rows', None)

import array

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import seaborn
seaborn.set(style='whitegrid')
seaborn.set_context('talk')

import plotly.graph_objects as go

import pickle


import copy

In [ ]:
# # Uma vez instalado, pode ser comentado
# !pip install python-igraph
# !pip install scikit-bio
# !pip install biopython
# !pip install ete3
# !pip3 install PyQt5
# !pip install python-Levenshtein
# !pip install matplotlib
# !pip install seaborn
# !pip install scikit-learn


In [ ]:
import importlib
import utils
importlib.reload(utils)

import phyil
importlib.reload(phyil)

In [ ]:
pwd

|# Preparacao dos dados

In [ ]:
df_dataset = pd.read_csv('dataset_v3.csv', sep=',')
df_original = df_dataset.copy()
# df.drop(columns=[
#     'cnae_2_0_classe',
#     'cnae_95_classe',
#     'ind_atividade_ano',
#     'ind_cei_vinculado',
#     'ind_estab_participa_pat',
#     'ind_rais_negativa',
#     'ind_simples',
#     'tamanho_estabelecimento',
#     'ibge_subsetor',
#     'id_1',
#     'qtd_vinculos_estatutarios'

# ], inplace=True)

# display(df)

In [ ]:
df_dataset.head()

In [ ]:
import ast

def string_to_list(s):
  try:
    return ast.literal_eval(s)
  except (ValueError, SyntaxError):
    return []

In [ ]:
# NAO NECESSARIO SE VAI USAR UMA DISTANCIA BASEADA EM STRING
# # Fransformacao das colunas de v_11 a v_43 para listas

# for col_num in np.arange(11,44,1):  # Assuming 'v_11' is the 11th column (index 10)]
#     col_name = f'v_{col_num}'
#     # Check if column exists before applying transformation
#     if col_name in df.columns:
#         # converter a string que representa uma lista em uma lista de numeros ou lista de numeros no formato string
#         df[col_name] = df[col_name].apply(string_to_list)
#         # Converter as listas de strings em listas de números
#         df[col_name] = df[col_name].apply(lambda x: [float(item) for item in x])
#         # Calcular a média dos elementos de cada coluna avaliada.
#         df[col_name] = df[col_name].apply(lambda x: sum(x) / len(x) if len(x) else 0)


# for col in ['qtd_vinculos_clt', 'qtd_vinculos_ativos', 'qtd_vinculos_estatutarios', 'ultraprocessado', 'misto']:
#     #  Check if column exists before applying transformation
#     if col in df.columns:
#         print(col)
#         # converter a string que representa uma lista em uma lista de numeros ou lista de numeros no formato string
#         df[col] = df[col].apply(string_to_list)
#         # Converter as listas de strings em listas de números
#         df[col] = df[col].apply(lambda x: [float(item) for item in x])
#         # Calcular a média dos elementos de cada coluna avaliada.
#         df[col] = df[col].apply(lambda x: sum(x) / len(x) if len(x) else 0)



# Analise multiObjetivo do dataframe X

In [ ]:
import array
import os

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import seaborn
seaborn.set(style='whitegrid')
seaborn.set_context('talk')
import pandas as pd
import copy

In [ ]:
markers = ['o', '^', 's', 'D', 'v', 'h', 'p']

In [ ]:
from deap import algorithms, base, benchmarks, tools, creator

# Implementação de um MOEA

In [ ]:
toolbox = base.Toolbox()

## Organização do df dos individuos com seu ID e apenas as colunas objetivos

In [ ]:
# df = pd.read_csv('dataset_v3.csv', sep=',')
df = df_dataset.copy()
dff = df[['numero_feiras_livres', 'ipvs', 'obj_in_natura']].copy()
# dff = df[['ipvs']].copy()
dff.head()

## binding the dataset with deap toolbox methods for individual creation and evaluation

In [ ]:
def already_evaluated_indivs(individual, df=dff):
  fvec = [ f for f in  df.loc[individual[0]] ]
  return fvec

In [ ]:
creator.create("Fitness3Obj", base.Fitness, weights=(+1.0,-1.0,+1.0))
# creator.create("Fitness3Obj", base.Fitness, weights=(-1.0,))
creator.create("Individual3Obj", array.array, typecode='i', fitness=creator.Fitness3Obj)
toolbox.register("evaluate", lambda ind: already_evaluated_indivs(ind))

In [ ]:
dff.shape[0]

In [ ]:
BOUND_LOW, BOUND_UP = 0, dff.shape[0]-1 # for already_evaluated_indiv()
N_DIM = 3
# N_DIM = 1

In [ ]:
def allintallele(bound_low, bound_up, size=None):
  i = bound_low
  while True:
    yield [i%bound_up]
    i += 1
allint = allintallele(BOUND_LOW, BOUND_UP+1, N_DIM)
def allintf(): #wraps generator allint as function, according to the format that toolbox.resgiter() requires
  return next(allint)

In [ ]:
toolbox.register("attr_float", allintf)
toolbox.register("individual", tools.initIterate,creator.Individual3Obj, toolbox.attr_float)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
toolbox.pop_size = dff.shape[0]
pop = toolbox.population(n=toolbox.pop_size)

In [ ]:
toolbox.register("select", tools.selNSGA2) # <====== NSGA2 SELECTION !!!

In [ ]:
toolbox.register("mate", tools.cxSimulatedBinaryBounded, low=BOUND_LOW, up=BOUND_UP, eta=20.0)
toolbox.register("mutate", tools.mutPolynomialBounded, low=BOUND_LOW, up=BOUND_UP, eta=20.0, indpb=1.0/N_DIM)

In [ ]:
def run_ea(toolbox, stats=None, verbose=False):
    pop = toolbox.population(n=toolbox.pop_size)
    pop = toolbox.select(pop, len(pop))
    return algorithms.eaMuPlusLambda(pop, toolbox, mu=toolbox.pop_size,
                                     lambda_=toolbox.pop_size,
                                     cxpb=1-toolbox.mut_prob,
                                     mutpb=toolbox.mut_prob,
                                     stats=stats,
                                     ngen=toolbox.max_gen,
                                     verbose=verbose)

In [ ]:
toolbox.max_gen  = 0 #Use zero to keep unchanged the current uploaded population
# toolbox.pop_size = 200 #10 #20 #100
toolbox.mut_prob = 0.2

## Running

In [ ]:
stats = tools.Statistics()
stats.register("pop", copy.deepcopy)
%time res, logbook = run_ea(toolbox, stats=stats)

In [ ]:
nsga2_sel = tools.selNSGA2(res,len(res))
nsga2_sel = tools.selNSGA2(res,len(res))
ev_allPoints = [ toolbox.evaluate(ind) for ind in nsga2_sel]#nsga2_sel or spea2_sel since both have all the points
df_all = pd.DataFrame(list(ev_allPoints), columns=dff.columns)
peeloff = [ g[0] for g in nsga2_sel ]
df_all['ID'] = peeloff
df_all['nsga2_rank'] = list(df_all.index+1)

In [ ]:
df_all.head()

In [ ]:
fronts = tools.sortNondominated(res, k=len(res))
# df_all['front'] = [ e for e, f in enumerate(fronts) for i in f ]#(i[0],e)
df_all['front'] = [ e for e, f in enumerate(fronts) for i in f ]#(i[0],e)
mf = len(fronts)
df_all['front color'] = [ mf-e for e, f in enumerate(fronts) for i in f ]

In [ ]:
df_all.head(10)

In [ ]:
df_all.to_csv(f'front_fome_NCD.csv', sep=';', decimal=',')

In [ ]:
dict_quantiles = {2: 0, 4: 1, 8: 2, 16: 3, 32: 4, 64: 5}

In [ ]:
df_all.head()

In [ ]:
df_all.nsga2_rank

In [ ]:
df_all.columns

# OBTENCAO DOS FIRST E LAST QUANTIS

In [ ]:
import pandas as pd
import numpy as np
import os

# --- PRÉ-REQUISITO: ---
# Certifique-se de que o DataFrame 'df_all' já esteja carregado e
# ORDENADO pelo critério desejado (ex: rank ou distância) antes de rodar este bloco.
# Exemplo: df_all = df_all.sort_values(by='nsga2_rank')

# Lista com as quantidades de quantis desejadas
quantis_config = [2, 4, 8, 16, 32, 64]

print("--- Iniciando a divisão e salvamento dos arquivos unificados ---")

for qnt_quantis in quantis_config:
    print(f"Processando para {qnt_quantis} quantis...")

    # 1. Divide o DataFrame em 'qnt_quantis' partes usando np.array_split
    # Isso substitui a função manual antiga. O numpy lida com restos de divisão automaticamente.
    lista_de_dfs_quantis = np.array_split(df_all, qnt_quantis)

    # 2. Adiciona a nova coluna 'quantil_idx' em cada parte
    lista_de_dfs_modificados = []
    for i, quantil_df in enumerate(lista_de_dfs_quantis):
        # Criamos uma cópia para evitar SettingWithCopyWarning do Pandas
        df_copia = quantil_df.copy()
        df_copia['quantil_idx'] = i
        lista_de_dfs_modificados.append(df_copia)

    # 3. Concatena todos os pedaços de volta em um único DataFrame
    df_final_com_quantis = pd.concat(lista_de_dfs_modificados, ignore_index=True)

    # 4. Define o nome da PASTA e do ARQUIVO e salva

    # Cria o diretório (ex: "4_quantis")
    output_dir = f'{qnt_quantis}_quantis'
    os.makedirs(output_dir, exist_ok=True)

    # Define o nome do arquivo (ex: "dados_divididos_em_4_quantis.csv")
    output_filename = f'dados_divididos_em_{qnt_quantis}_quantis.csv'

    # Junta o caminho da pasta e o nome do arquivo
    file_path = os.path.join(output_dir, output_filename)

    # Salva o DataFrame final
    df_final_com_quantis.to_csv(file_path, index=False, sep=';', decimal=',')

    print(f"  -> Arquivo salvo em: {file_path}")

print("\nProcesso finalizado com sucesso!")

# OBTENCAO DE TODOS OS QUANTIS

In [ ]:
import pandas as pd
import numpy as np
import os

# --- Parte 2: Sua Função Original para Obter os Quantis (sem alterações) ---

def get_all_quantis(df, qnt_quantis):
    """
    Divide um DataFrame em um número específico de partes (quantis posicionais).
    """
    qnt_samples = df.shape[0]
    quantis_list = []
    size_quantil = int(qnt_samples / qnt_quantis)

    for i in range(qnt_quantis):
        start_idx = i * size_quantil
        if i == qnt_quantis - 1:
            quantil = df.iloc[start_idx:]
        else:
            quantil = df.iloc[start_idx:start_idx + size_quantil]
        quantis_list.append(quantil)

    return quantis_list

# --- Parte 3: Lógica CORRIGIDA para Salvar Arquivo Único e os Extremos ---

# Lista com as quantidades de quantis desejadas
quantis_config = [2, 4, 8, 16, 32, 64]

for qnt_quantis in quantis_config:
    print(f"--- Processando para {qnt_quantis} quantis ---")

    # 1. Gera a lista de DataFrames
    lista_de_dfs_quantis = get_all_quantis(df_all, qnt_quantis)

    # 2. Adiciona a nova coluna 'quantil_idx' em cada DataFrame da lista
    lista_de_dfs_modificados = []
    for i, quantil_df in enumerate(lista_de_dfs_quantis):
        df_copia = quantil_df.copy()
        df_copia['quantil_idx'] = i
        lista_de_dfs_modificados.append(df_copia)

    # 3. Concatena todos os DataFrames em um só
    df_final_com_quantis = pd.concat(lista_de_dfs_modificados, ignore_index=True)

    # 4. Define o nome da PASTA e salva os arquivos no local correto
    # Cria o diretório (ex: "4_quantis")
    output_dir = f'{qnt_quantis}_quantis'
    os.makedirs(output_dir, exist_ok=True)

    # ==============================================================
    # LÓGICA ADICIONADA: Salvar o primeiro e o último quantil
    # ==============================================================
    df_first = lista_de_dfs_modificados[0]
    df_last = lista_de_dfs_modificados[-1]

    # Caminhos para os novos arquivos
    file_path_first = os.path.join(output_dir, f'first_quantil_{qnt_quantis}.csv')
    file_path_last = os.path.join(output_dir, f'last_quantil_{qnt_quantis}.csv')

    # Salvando os arquivos
    df_first.to_csv(file_path_first, index=False, sep=';', decimal=',')
    df_last.to_csv(file_path_last, index=False, sep=';', decimal=',')

    print(f"Arquivos extremos salvos: {file_path_first} e {file_path_last}")
    # ==============================================================

    # Define o nome do arquivo unificado (ex: "dados_divididos_em_4_quantis.csv")
    output_filename = f'dados_divididos_em_{qnt_quantis}_quantis.csv'

    # Junta o caminho da pasta e o nome do arquivo unificado
    file_path = os.path.join(output_dir, output_filename)

    # Salva o DataFrame final no caminho completo
    df_final_com_quantis.to_csv(file_path, index=False, sep=';', decimal=',')

    print(f"Arquivo único salvo em: {file_path}")
    print("-" * 30 + "\n")

print("Processo finalizado com sucesso!")

In [ ]:
# df_total = pd.read_csv('dist_df_ncd_15_vizim_lzma.csv', sep=';', decimal=',')
# df_total.drop(columns=['Unnamed: 0'], inplace=True)
# df_total.head()

# Carregamento das listas first_quantis e last_quantis

In [ ]:
# CARREGAMENTO DOS FIRSTS E LASTS QUANTIS
# --- 1. CONFIGURAÇÃO ---

# Lista de quantis para procurar as pastas e arquivos. A ordem aqui definirá a ordem nas listas de resultado.
quantis = [2, 4, 8, 16, 32, 64]

# Diretório raiz onde as pastas '{qq}_quantis' estão localizadas.
diretorio_raiz = os.getcwd()

# ALTERADO: Inicializa duas listas vazias.
first_quantis = []
last_quantis = []

print(f"Iniciando a busca e organização de arquivos em listas: {diretorio_raiz}")

# --- 2. LOOP PRINCIPAL PARA LER OS ARQUIVOS E ADICIONAR ÀS LISTAS ---

for qq in tqdm(quantis, desc="Processando quantis"):
    # Constrói o caminho para a pasta do quantil atual
    folder_path = os.path.join(diretorio_raiz, f'{qq}_quantis')

    # --- Processa o arquivo 'first_quantil' ---
    first_filepath = os.path.join(folder_path, f'first_quantil_{qq}.csv')

    if os.path.exists(first_filepath):
        try:
            # Lê o DataFrame
            df_first = pd.read_csv(first_filepath, sep=';', decimal=',')
            # ALTERADO: Adiciona o DataFrame à lista 'first_quantis'
            first_quantis.append(df_first)
        except Exception as e:
            print(f"\nOcorreu um erro ao ler o arquivo {first_filepath}: {e}")
            first_quantis.append(None) # Adiciona None em caso de erro de leitura
    else:
        print(f"\nAVISO: Arquivo 'first' não encontrado para qq={qq}, adicionando None à lista.")
        # ALTERADO: Adiciona None para manter o alinhamento dos índices
        first_quantis.append(None)

    # --- Processa o arquivo 'last_quantil' ---
    last_filepath = os.path.join(folder_path, f'last_quantil_{qq}.csv')

    if os.path.exists(last_filepath):
        try:
            df_last = pd.read_csv(last_filepath, sep=';', decimal=',')
            # ALTERADO: Adiciona o DataFrame à lista 'last_quantis'
            last_quantis.append(df_last)
        except Exception as e:
            print(f"\nOcorreu um erro ao ler o arquivo {last_filepath}: {e}")
            last_quantis.append(None)
    else:
        print(f"\nAVISO: Arquivo 'last' não encontrado para qq={qq}, adicionando None à lista.")
        # ALTERADO: Adiciona None para manter o alinhamento dos índices
        last_quantis.append(None)

# --- 3. VERIFICAÇÃO FINAL ---
print("\n--- Processo de Carregamento Concluído ---")

# ALTERADO: Verificação para a lista 'first_quantis'
print("\nResumo dos dados carregados na lista 'first_quantis':")
for i, df in enumerate(first_quantis):
    qq_correspondente = quantis[i]
    if df is not None:
        print(f"  - Posição {i} (qq={qq_correspondente}) carregada. Dimensões: {df.shape}")
    else:
        print(f"  - Posição {i} (qq={qq_correspondente}) não foi carregada (arquivo não encontrado ou erro).")

# ALTERADO: Verificação para a lista 'last_quantis'
print("\nResumo dos dados carregados na lista 'last_quantis':")
for i, df in enumerate(last_quantis):
    qq_correspondente = quantis[i]
    if df is not None:
        print(f"  - Posição {i} (qq={qq_correspondente}) carregada. Dimensões: {df.shape}")
    else:
        print(f"  - Posição {i} (qq={qq_correspondente}) não foi carregada (arquivo não encontrado ou erro).")

# Obtenção da matriz de distância dos quantis que serão foco da análise


In [ ]:
df.head()

In [ ]:
# saida = phyil.process_distance_matrices(df, first_quantis, last_quantis, dict_quantiles,
#                             metric='levenshtein', num_vizim=10,qnt_quantis=qq, distance_matrix = False)

In [ ]:
# producao dos arquivos df_foco e dist_df para valores de qq variados e FIRST e LAST quantis.
# Se o arquivo já tiver sido produzido, pode comentar essa parte.

# É retornado um dict com diversas informacoes de distancia a cerca
# dos quantils FIRST e LAST para uma qnt de quantil especifica.

#         Dicionário com duas chaves: 'FIRST' e 'LAST'. Cada valor é um sub-dicionário contendo:
#             - df_foco               : pandas.DataFrame com as linhas selecionadas de `df`.
#             - dist_df               : pandas.DataFrame da matriz de distância normalizada.
#             - dist_df_geodesica     : pandas.DataFrame da matriz de distância geodésica.
#             - geodesic_knn_sparse   : matriz esparsa de adjacência do grafo geodésico.
#             - geodesic_knn          : lista ou estrutura completa do grafo geodésico.
#             - geodesic_dist_matrix  : numpy.ndarray da matriz de distâncias geodésicas.

df = df_dataset.copy()
for qq in [2,4,8,16]:
    # retorna apenas df_foco se distance_matrix for False
    saida = phyil.process_distance_matrices(df, first_quantis, last_quantis, dict_quantiles,
                            metric='levenshtein', num_vizim=10,qnt_quantis=qq, distance_matrix = False)

    for tipo in ['first', 'last']:

        # sao retiradas do dataset os objetivos e informacoes que nao caracterizam as amostras
        # como o index, cod_dist e nome_area
        df_foco = saida[tipo.upper()]['df_foco'].drop(columns=['numero_feiras_livres', 'ipvs', 'obj_in_natura', 'cod_dist', 'nome_area'])

        # Definição do nome dos arquivos que serao salvos
        filename_foco = f'df_foco_{tipo}_{qq}_quantis.csv'
        filename_dist = f'dist_df_{tipo}_{qq}_quantis.csv'

        # salvamento do slice do df_all referente ao tipo
        df_foco.to_csv(filename_foco, sep=';', decimal=',')

        # calculo da matriz de distncia do df_foco
        if "levenshtein":
            matriz_dist_all_dataset = utils.nld_distance_matrix_parallel(df_foco.iloc[:,1:])
        df_dist = pd.DataFrame(
                                matriz_dist_all_dataset,
                                index=df_foco.index,
                                columns=df_foco.index
                               )
        # salvamento da matriz de distancia refetne ao df_foco
        df_dist.to_csv(filename_dist, sep=';', decimal=',')

In [ ]:
saida['FIRST']['df_foco']

# Obtenção da matriz de distância de todo o dataset

In [ ]:
df_dataset.head()

In [ ]:
# --- Código 1 AJUSTADO para normalizar (como no Código 2) ---

# usar quando nao quiser gerar toda vez a matriz de distancia
filename = 'matrizDistanciaLevenshteinBaseInteira.csv'

if not os.path.exists(filename):
    # Arquivo não existe: executa o cálculo, NORMALIZA e salva
    print(f'Arquivo "{filename}" não encontrado. Calculando e normalizando...')

    # deixando apenas as colunas informativas e que nao sao objetivos
    df_base = df_dataset.drop(columns = ['numero_feiras_livres', 'ipvs', 'obj_in_natura', 'cod_dist', 'nome_area'], errors='ignore').copy()

    # Calcula a matriz de distância original
    if "levenshtein":
        matriz_dist_all_dataset = utils.nld_distance_matrix_parallel(df_base)
        # matriz_dist_all_dataset = utils.matrizDistancia(df_base.to_numpy(), 'euclidiana', 2)
    df_dist_original = pd.DataFrame(
                            matriz_dist_all_dataset,
                            index=df_base.index,
                            columns=df_base.index
                           )

    # --- Lógica de Normalização copiada do Código 2 ---
    try:
        print("Normalizando a matriz de distância...")
        # Assumindo que 'phyil' foi importado e 'normalize_df_0_1' existe
        df_dist_normalizada = phyil.normalize_df_0_1(df_dist_original)
        print("Normalização concluída.")

        # Salva a matriz NORMALIZADA
        df_dist_normalizada.to_csv(filename, sep=';', decimal=',')
        print(f'Arquivo normalizado "{filename}" gerado e salvo.')

    except ValueError as e:
        print(f"ERRO durante a normalização: {e}")
        print("Salvando a matriz de distância original sem normalização.")
        # Salva a original com um nome de backup em caso de falha
        df_dist_original.to_csv('matrizDistanciaDTLZ2BaseInteira_ORIGINAL_FALHA_NORM.csv', sep=';', decimal=',')
    except AttributeError as ae:
        print(f"ERRO: A função 'phyil.normalize_df_0_1' não foi encontrada. {ae}")
        print("Salvando a matriz de distância original sem normalização.")
        df_dist_original.to_csv('matrizDistanciaDTLZ2BaseInteira_ORIGINAL_FALHA_NORM.csv', sep=';', decimal=',')


else:
    # Arquivo já existe: pula a execução
    print(f'Arquivo "{filename}" já existe. Operação não executada.')

# Filograma e Detecção de Comunidade

# Unificacao do codigo.

## Geração dos arquivos df_rotulados

In [ ]:
for qq in tqdm([2,4,8,16]):
    print(f'\n{qq} quantis')
    # se o algoritmo nao for definido, entao eh o label spreading com alpha = 0.1
    df_temp = phyil.rotular_por_quantil(qq, df_original.copy(), phyil.load_results_from_csv, utils, sigma=0.10, matriz_dist_path = filename)

    # Salvar a tupla completa em um arquivo pickle
    # Arquivo conterá o df_foco, amostras rotuladas dos quantis first e last, etc.
    with open(f'dados_rotulados_{qq}_quantis.pkl', 'wb') as f:
        pickle.dump(df_temp, f)

    # Salvar o dataframe com as informacoes da rotulacao
    df_saida = df_temp[0]
    df_saida.to_csv(f'df_rotulado_{qq}_quantis.csv', sep=';', decimal=',')

# Quando for ler use os comandos abaixo
# with open('saida_completa.pkl', 'rb') as f:
# dados_carregados = pickle.load(f)

### contagem de categorias para df ja rotulados

In [ ]:
# #  mesma coisa da celula anterior mas obtem os vetores de quantidade
# # # celula utilizada para ler um dataframe que ja foi rotulado e contar as categorias.
# # # como ja foi salvo, o df ja tem as informacoes de convergencia e valores de margem e entropia.
# import pandas as pd
#
# quantis = [2, 4, 8, 16]
# categorias = ['Confiável', 'Confiança Fraca', 'Confiança Média']
#
# # Inicialize as listas
# qnt_conf_Forte_PorQuantil_margem = []
# qnt_conf_Fraca_PorQuantil_margem = []
# qnt_conf_Media_PorQuantil_margem = []
#
# qnt_conf_Forte_PorQuantil_entropia = []
# qnt_conf_Fraca_PorQuantil_entropia = []
# qnt_conf_Media_PorQuantil_entropia = []
#
# for qq in quantis:
#     try:
#         df = pd.read_csv(f'df_rotulado_{qq}_quantis.csv', sep=';', decimal=',', index_col=0)
#     except FileNotFoundError:
#         print(f'Arquivo df_rotulado_{qq}_quantis.csv não encontrado.\n')
#         # Para manter o alinhamento das listas, adicione 0 se o arquivo não existir
#         qnt_conf_Forte_PorQuantil_margem.append(0)
#         qnt_conf_Fraca_PorQuantil_margem.append(0)
#         qnt_conf_Media_PorQuantil_margem.append(0)
#         qnt_conf_Forte_PorQuantil_entropia.append(0)
#         qnt_conf_Fraca_PorQuantil_entropia.append(0)
#         qnt_conf_Media_PorQuantil_entropia.append(0)
#         continue
#
#     # Margem
#     contagem_margem = df[f'status - Margem {qq} quantis'].value_counts()
#     qnt_conf_Forte_PorQuantil_margem.append(contagem_margem.get('Confiança Forte', 0))
#     qnt_conf_Fraca_PorQuantil_margem.append(contagem_margem.get('Confiança Fraca', 0))
#     qnt_conf_Media_PorQuantil_margem.append(contagem_margem.get('Confiança Média', 0))
#
#     # Entropia
#     contagem_entropia = df[f'status - Entropia {qq} quantis'].value_counts()
#     qnt_conf_Forte_PorQuantil_entropia.append(contagem_entropia.get('Confiança Forte', 0))
#     qnt_conf_Fraca_PorQuantil_entropia.append(contagem_entropia.get('Confiança Fraca', 0))
#     qnt_conf_Media_PorQuantil_entropia.append(contagem_entropia.get('Confiança Média', 0))
#
# # Exemplo de impressão dos resultados
# print("Margem - Confiança Forte:", qnt_conf_Forte_PorQuantil_margem)
# print("Margem - Confiança Fraca:", qnt_conf_Fraca_PorQuantil_margem)
# print("Margem - Confiança Média:", qnt_conf_Media_PorQuantil_margem)
# print("Entropia - Confiança Forte:", qnt_conf_Forte_PorQuantil_entropia)
# print("Entropia - Confiança Fraca:", qnt_conf_Fraca_PorQuantil_entropia)
# print("Entropia - Confiança Média:", qnt_conf_Media_PorQuantil_entropia)


In [ ]:
# df[['predicted_label 2 quantis', 'status - Margem 2 quantis',
#        'status - Entropia 2 quantis', 'predicted_label 4 quantis',
#        'status - Margem 4 quantis', 'status - Entropia 4 quantis',
#        'predicted_label 8 quantis', 'status - Margem 8 quantis',
#        'status - Entropia 8 quantis', 'predicted_label 16 quantis',
#        'status - Margem 16 quantis', 'status - Entropia 16 quantis']]

In [ ]:
# plot_percent_by_quantil(
    # qnt_conf_Media_PorQuantil_entropia, qnt_conf_Forte_PorQuantil_entropia, qnt_conf_Fraca_PorQuantil_entropia, len(df), quantis=[2, 4, 8, 16], width=900, height=600, cat='Entropia')

In [ ]:
# plot_percent_by_quantil(
#     qnt_conf_Media_PorQuantil_margem, qnt_conf_Forte_PorQuantil_margem, qnt_conf_Fraca_PorQuantil_margem, len(df), quantis=[2, 4, 8, 16], width=900, height=600, cat= 'Margem')

# Codigo com variacao do valor de sigma na matriz de similaridade.
## este codigo compoe todos os codigos acima.

In [ ]:
df_original.columns

In [ ]:
df.columns.equals(df_original.columns)

In [ ]:
for q in [2,4,8,16]:
  print(f'\n{q} quantis')
  df_temp = phyil.rotular_por_quantil(q, df_original.copy(), phyil.load_results_from_csv, utils, sigma=1)

# CODIGO PARA APAGAR ARQUIVOS

In [ ]:
# # prompt: mostre um script python para apagar arquivos de uma pasta do drive que comecam com uma string

# import os

# def delete_files_starting_with(folder_path, prefix):
#   """Deletes files in a folder that start with a given prefix.

#   Args:
#     folder_path: The path to the folder.
#     prefix: The prefix string.
#   """
#   for filename in os.listdir(folder_path):
#     if filename.startswith(prefix):
#       file_path = os.path.join(folder_path, filename)
#       try:
#         if os.path.isfile(file_path):
#           os.remove(file_path)
#           print(f"Deleted file: {file_path}")
#         else:
#           print(f"Skipping directory: {file_path}")
#       except OSError as e:
#         print(f"Error deleting file {file_path}: {e}")

# # Example usage:
# folder_path = '/content/drive/MyDrive/Doutorado/analiseResultadosRegrasAssociacao' # Replace with your folder path
# prefix_to_delete = 'df_rotulado_16_quantis_sigma'  # Replace with the prefix you want to use

# delete_files_starting_with(folder_path, prefix_to_delete)


# Geração de matrizes pertubadas

## Geracao das margens de n matrizes pertubadas e m pertubacoes cada
### margens ja obtidas. Descomentar se precisar cria-las novamente

In [ ]:
import multiprocessing
import os
import numpy as np

# --- Passo 1: Bloco principal para orquestrar os processos ---
if __name__ == '__main__':
    # 1. Defina os parâmetros comuns
    algoritmo = 'label_spreading'
    alpha = 0.1

    qq_valores = [2, 4, 8, 16]
    sigmas = np.arange(0.01, 0.31, 0.01)

    # qq_valores = [64]
    # sigmas = np.arange(0.01, 0.02, 0.01)
    num_matriz_Pert = 100
    num_pert = 200
    output_dir_base = 'matrizes_margens_perturbadas'
    os.makedirs(output_dir_base, exist_ok=True)
    matriz_dist_path = os.path.join(os.getcwd(), 'matrizDistanciaLevenshteinBaseInteira.csv')
    sigma_noise = 0.5
    save = True

    processos = []

    # --- Passo 2: Criar e iniciar um processo para cada valor de qq ---
    for qq in tqdm(qq_valores, desc='Processos - {qq}'):
        # Agora o 'target' usa a função que foi importada
        processo = multiprocessing.Process(
            target=phyil.processar_qq,
            args=(
                qq, df_original, algoritmo, alpha, sigmas,
                num_matriz_Pert, num_pert, output_dir_base,
                matriz_dist_path, sigma_noise, save
            )
        )
        processos.append(processo)
        processo.start()

    # --- Passo 3: Esperar que todos os processos terminem ---
    for processo in processos:
        processo.join()

    print("\nTodos os processos foram concluídos.")

### Matrizes de similaridade ja obtidas. descomente se precisar rodar novamente.

In [ ]:
# for qq in tqdm([2,4,8,16]):
#     FOLDER_PATH = f'G:\\Meu Drive\\Doutorado\\analiseResultadosRegrasAssociacao\\matrizes_similaridade\\{qq}_quantis'
#     os.makedirs(FOLDER_PATH, exist_ok=True)
#     for sigma in tqdm(np.arange(0.01, 0.31, 0.01)):
#         filename = f'matriz_similaridade_{sigma:.2f}_sigma_{qq}_quantis.csv'
        
#         phyil.salvar_matriz_similaridade(
#             matriz_dist_path = 'matrizDistanciaLevenshteinBaseInteira.csv', 
#             sigma= sigma, 
#             matriz_sim_path=os.path.join(FOLDER_PATH, filename)
#         )


# Concatenação das margens pertubadas
### Já feito, se precisar rodar novamente, descomente.

In [ ]:
# Pega todos os arquivos (vetores) com as margens para um valor de quantil e cria uma matriz. Depois salva-a
# O valor de alpha eh fixo pois ao usar o label spreading, o alpha nao eh alterado.
alpha = 0.10
for qq in tqdm([2, 4, 8, 16]):
# for qq in tqdm([2]):
    # 1. Parâmetros
    # FOLDER_PATH = '/content/drive/MyDrive/Doutorado/analiseResultadosRegrasAssociacao/matrizes_margens_perturbadas_temp' # se colab

    FOLDER_PATH = f'{dest}\\matrizes_margens_perturbadas\\{qq}_quantis'
    FOLDER_PATH_dest = f'{dest}\\{qq}_quantis'
    # print(FOLDER_PATH)
    algoritmo  = 'label_spreading'
    # sigmas     = [0.01, 0.02]
    sigmas     = np.arange(0.01, 0.31, 0.01)

    # 2. Acumula todos os DataFrames aqui
    all_dfs = []

    for sigma in tqdm(sigmas):
        # 3. Monta pattern para encontrar apenas os CSVs deste sigma
        #    Use f'{sigma:.2f}' se seus arquivos usarem sempre duas casas

        if algoritmo == 'label_spreading':
            pattern = os.path.join(
                FOLDER_PATH,
                f'matriz_margens_completa_{qq}_quantis_{algoritmo}_sigma_{sigma:.2f}_alpha_{alpha:.2f}_pert_*.csv'
            )
        else:
            pattern = os.path.join(
                FOLDER_PATH,
                f'matriz_margens_completa_{qq}_quantis_{algoritmo}_sigma_{sigma:.2f}_pert_*.csv'
            )
        files = sorted(glob.glob(pattern))
        # print(f'qnt de files: {len(files)}')
        if not files:
            print(f'Nenhum arquivo encontrado para sigma={sigma:.2f}')
            continue

        # 4. Para cada arquivo, leia e acrescente colunas de controle
        for path in files:
            df = pd.read_csv(path, sep=';', decimal=',')

            # Extrai o número da perturbação do nome:
            # exemplo: ..._sigma_0.05_pert_12.csv → '12'
            pert_str = os.path.basename(path).split('_pert_')[-1]
            perturbacao = int(pert_str.replace('.csv', ''))

            df['perturbacao'] = perturbacao
            df['sigma']       = round(sigma, 2)
            all_dfs.append(df)

    # 5. Concatena tudo num único DataFrame
    if all_dfs:
        df_total = pd.concat(all_dfs, ignore_index=True)
    else:
        df_total = pd.DataFrame()  # vazio se não achou nada

    # 6. (Opcional) Salva num CSV consolidado
    output_path = os.path.join(FOLDER_PATH, f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv')
    df_total.to_csv(output_path, sep=';', decimal=',', index=False)

    # 7. Inspeção rápida
    print(f'Dados para {qq} quantis')
    display(df_total.shape)
    display(df_total.head())

# Verificação de meios de validacao da rotulacao
abaixo faz-se um grafico de superficie com as margens obtidas para cada pertubacao

In [ ]:
df_dataset.to_csv('df_dataset.csv', sep=';', decimal=',', index=False)
df_all.to_csv('df_all.csv', sep=';', decimal=',', index=False)

### Histograma para cada quantil para um valor especifico de sigma

In [ ]:
# for qq in [2,4,8,16]:
#
#     # qq = 16
#     FOLDER_PATH = f'G:\\Meu Drive\\Doutorado\\analiseResultadosRegrasAssociacao\\matrizes_margens_perturbadas\\{qq}_quantis'
#     filename = f'matriz_margens_todas_perturbacoes_todos_sigmas_{qq}_quantis.csv'
#     df = pd.read_csv(os.path.join(FOLDER_PATH, filename), sep=';', decimal=',')
#
#     df = df[df['sigma'] == 0.15]
#
#     # Derretendo o DataFrame para formato "long"
#     df_long = df.melt(
#         id_vars=['sigma','perturbacao'],
#         value_vars=[f'margem_{i}' for i in range(1,97)],
#         var_name='amostra',
#         value_name='margem'
#     )
#
#     def categoria(m):
#         if m < 0.2:   return 'Fraca'
#         if m > 0.6:   return 'Forte'
#         return 'Média'
#
#     df_long['categoria'] = df_long['margem'].map(categoria)
#
#     # Gráfico de barras de proporção usando Plotly Express
#     fig = px.histogram(
#         df_long,
#         x='categoria',
#         color='categoria',
#         category_orders={'categoria': ['Fraca', 'Média', 'Forte']},
#         color_discrete_sequence=px.colors.qualitative.Set2,
#         text_auto=True
#     )
#
#     fig.update_layout(
#         title=f'Contagem de estado de confiança (todas perturbações) - {qq} quantis',
#         xaxis_title='Categoria',
#         yaxis_title='Contagem',
#         showlegend=False
#     )
#
#     fig.show()


### Boxplot para as pertubacoes de cada amostra para um valor de sigma

### Validação por pertubacao

### Cópia de arquivos das matrizes de similaridades
Faz uma copia das matrizes de similaridade para uma unica pasta chamada matrizes_similaridades. Elas nao sao recalculadas, elas sao apenas copiadas. Apos a copia pode-se comentar o codigo abaixo

In [ ]:
import os
import shutil
import re

# --- CONFIGURAÇÃO ---
# O script usará o diretório atual como raiz.
diretorio_raiz = os.getcwd()

# --- LÓGICA DO SCRIPT ---

# Nome da nova pasta de destino principal
pasta_destino_principal = "matrizes_similaridades"

# Cria o caminho completo para a nova pasta principal
caminho_destino_principal = os.path.join(diretorio_raiz, pasta_destino_principal)

# Cria a pasta "matrizes_similaridade" se ela não existir
try:
    os.makedirs(caminho_destino_principal, exist_ok=True)
    print(f"Pasta principal de destino em: '{caminho_destino_principal}'")
except OSError as e:
    print(f"Erro ao criar o diretório principal: {e}")
    exit() # Encerra o script se não for possível criar a pasta

print("\n--- Iniciando a busca e cópia de arquivos ---")

arquivos_copiados_total = 0

# os.walk percorre de forma recursiva todas as pastas e arquivos
# a partir do diretório raiz.
for pasta_atual, subpastas, arquivos in os.walk(diretorio_raiz):

    # Pula a própria pasta de destino para evitar loops infinitos ou cópias indesejadas
    if pasta_destino_principal in pasta_atual:
        continue

    # Itera sobre todos os arquivos encontrados na pasta atual
    for nome_arquivo in arquivos:
        # Verifica se o nome do arquivo começa com o prefixo desejado
        if nome_arquivo.startswith("matriz_similaridade_original"):

            # Pega apenas o nome da pasta onde o arquivo está
            nome_pasta_origem = os.path.basename(pasta_atual)

            # Tenta extrair o número do nome da pasta (ex: de "fome_2_quantis" extrai "2")
            numeros_encontrados = re.findall(r'\d+', nome_pasta_origem)

            if not numeros_encontrados:
                print(f"AVISO: Nenhum número encontrado no nome da pasta '{nome_pasta_origem}'. Pulando arquivo '{nome_arquivo}'.")
                continue

            # Pega o primeiro número encontrado no nome da pasta
            numero_quantil = numeros_encontrados[0]

            # Monta o nome da subpasta de destino, ex: "2_quantis"
            nome_pasta_destino_sub = f"{numero_quantil}_quantis"
            caminho_pasta_destino_sub = os.path.join(caminho_destino_principal, nome_pasta_destino_sub)

            # Cria a subpasta de destino se ela não existir
            os.makedirs(caminho_pasta_destino_sub, exist_ok=True)

            # Monta o caminho completo do arquivo de origem e de destino
            caminho_arquivo_origem = os.path.join(pasta_atual, nome_arquivo)
            caminho_arquivo_destino = os.path.join(caminho_pasta_destino_sub, nome_arquivo)

            # Copia o arquivo da origem para o destino
            try:
                shutil.copy2(caminho_arquivo_origem, caminho_arquivo_destino)
                print(f"  -> Copiado '{nome_arquivo}' de '{nome_pasta_origem}' para '{nome_pasta_destino_sub}'")
                arquivos_copiados_total += 1
            except Exception as e:
                print(f"  -> ERRO ao copiar '{nome_arquivo}': {e}")


if arquivos_copiados_total == 0:
    print("\nNenhum arquivo 'matriz_similaridade_original' foi encontrado nas subpastas.")
else:
    print(f"\n--- Processo concluído! Total de {arquivos_copiados_total} arquivos copiados. ---")

In [ ]:
phyil.difusao_rotulos

In [ ]:
pwd

### Producao dos Flip Rates
#### Ja obtido os flip rates. Descomente para calcular novamente.

### Abaixo sao calculados e salvos os rotulos para cada matriz pertubada e para a original. Após obter os resultados, nao precisa rodar novamente. Entao o trecho pode ser comentado.

In [ ]:
df_dataset.head()

In [ ]:
from multiprocessing import Pool, cpu_count
if __name__ == "__main__":

    try:

        df_dataset = pd.read_csv('df_dataset.csv', sep=';', decimal=',')

    except FileNotFoundError:
        print("Arquivo de dados não encontrado!")


    # --- Parâmetros Globais ---
    matriz_distance_name = 'matrizDistanciaLevenshteinBaseInteira.csv'
    df_original = df_dataset.copy()
    algoritmo = 'label_spreading'
    sigmaNoise = 0.5

    # Parâmetros para continuar a execução em caso de erro
    qq_inicial = 2
    sigma_inicial = 0.01

    # Lista de todos os valores de 'qq' a serem processados
    lista_qq = [2, 4, 8, 16]

    # Prepara os argumentos para cada processo.
    # Cada processo recebe uma tupla com todos os parâmetros necessários.
    tasks = []
    for i, qq_val in enumerate(lista_qq):
        tasks.append((
            qq_val,
            df_original.copy(),
            matriz_distance_name,
            algoritmo,
            sigmaNoise,
            qq_inicial,
            sigma_inicial,
            i  # Posição para a barra de progresso
        ))

    print(f"Iniciando o processamento paralelo em {min(len(lista_qq), cpu_count())} cores...")

    # Cria um "Pool" de processos. O número de processos é o número de 'qq's
    # ou o número de CPUs disponíveis, o que for menor.
    with Pool(processes=min(len(lista_qq), cpu_count())) as pool:
        # O 'pool.map' distribui a função 'processar_qq' para cada item na lista 'tasks'
        # e aguarda a conclusão de todos.
        results = pool.map(phyil.processar_rotulos_qq, tasks)

    print("\n" + "="*50)
    for result in results:
        print(result)
    print("Processamento paralelo concluído!")


## Geração dos flip_rates a partir dos arquivos de rotulacao ja obtidos.
Comentar se ja executado. Descomentar se quiser rodar novamente.

In [ ]:
# qq = 64
for qq in [2,4,8,16]:
    # Define o nome do arquivo de entrada
    FOLDER_PATH = os.path.join(os.getcwd(), f'{qq}_quantis')
    arquivo_com_rotulos = os.path.join(FOLDER_PATH, f'resultados_rotulos_{qq}_quantis.csv')

    # Exemplo 2: Chamar a função especificando um nome para o arquivo de saída.
    nome_customizado = os.path.join(FOLDER_PATH, f'flip_rate_{qq}_quantis.csv')
    phyil.calcular_flip_rate_de_arquivo(arquivo_com_rotulos, arquivo_de_saida=nome_customizado)

## codigo usado para copiar os arquivos de flip rate para uma unica pasta.

In [ ]:
# 1. Parâmetros de configuração
# Lista de valores de 'qq' que definem as pastas de origem.
lista_qq = [2, 4, 8, 16]

# Nome da pasta de destino para onde os arquivos serão copiados.
pasta_destino = 'flip_rates'

# Extensão esperada para os arquivos. Mude se for diferente (ex: '.txt').
extensao_arquivo = '.csv'

# Contador para o relatório final
arquivos_copiados = 0
arquivos_faltantes = 0

print(f"--- INICIANDO A BUSCA E CÓPIA DE ARQUIVOS ---")

# 2. Cria a pasta de destino, se ela ainda não existir
# O argumento 'exist_ok=True' previne um erro caso a pasta já exista.
print(f"Verificando/Criando a pasta de destino: '{pasta_destino}'...")
os.makedirs(pasta_destino, exist_ok=True)

# 3. Loop principal para percorrer cada valor de 'qq'
for qq in lista_qq:
    # Monta dinamicamente o nome da pasta e do arquivo de origem
    pasta_origem = f'{qq}_quantis'
    nome_arquivo = f'flip_rate_{qq}_quantis{extensao_arquivo}'

    # Cria o caminho completo para o arquivo de origem
    # os.path.join() garante que o caminho seja válido em qualquer sistema operacional
    caminho_origem = os.path.join(pasta_origem, nome_arquivo)

    print(f"\nBuscando em '{pasta_origem}'...")

    # 4. Verifica se o arquivo de origem realmente existe
    if os.path.exists(caminho_origem):
        # Se existe, define o caminho de destino
        caminho_destino = os.path.join(pasta_destino, nome_arquivo)

        print(f"  -> Arquivo encontrado: '{caminho_origem}'")

        try:
            # Copia o arquivo da origem para o destino.
            # shutil.copy2() também copia metadados do arquivo (como data de modificação).
            shutil.copy2(caminho_origem, caminho_destino)
            print(f"  -> Copiado com sucesso para '{caminho_destino}'")
            arquivos_copiados += 1
        except Exception as e:
            print(f"  -> ERRO ao copiar o arquivo: {e}")

    else:
        # Se não existe, exibe um aviso
        print(f"  -> AVISO: Arquivo '{caminho_origem}' não encontrado. Pulando.")
        arquivos_faltantes += 1

# 5. Relatório final da operação
print("\n" + "="*50)
print("--- PROCESSO CONCLUÍDO ---")
print(f"Total de arquivos copiados: {arquivos_copiados}")
print(f"Total de arquivos não encontrados: {arquivos_faltantes}")
print(f"Todos os arquivos encontrados foram copiados para a pasta '{pasta_destino}'.")
print("="*50)

Unificacao dos flip rates em um unico arquivo. Ja obtido. Descomente para calcular novamente.

In [ ]:
# ==============================================================================
# SCRIPT PRINCIPAL PARA UNIFICAR ARQUIVOS CSV JÁ EXISTENTES
# ==============================================================================

# 1. Parâmetros de configuração
# ATENÇÃO: Altere este nome para o da pasta onde estão seus arquivos CSV de flip rate.
diretorio_dos_arquivos = 'flip_rates'

# Nome do arquivo CSV de saída que conterá todos os resultados combinados.
arquivo_final_unificado = 'flip_rates_consolidados.csv'

# Lista para armazenar os DataFrames lidos de cada arquivo.
lista_de_dataframes = []

print(f"--- INICIANDO A UNIFICAÇÃO DE ARQUIVOS DE FLIP RATE ---")

# 2. Verifica se o diretório de entrada existe
if not os.path.isdir(diretorio_dos_arquivos):
    print(f"\nERRO CRÍTICO: O diretório '{diretorio_dos_arquivos}' não foi encontrado.")
    print("Por favor, verifique o nome da pasta ou o local deste script.")
else:
    print(f"Procurando por arquivos .csv no diretório: '{diretorio_dos_arquivos}'\n")

    # 3. Loop para ler cada arquivo no diretório especificado
    # os.listdir() retorna uma lista com o nome de todos os arquivos e pastas.
    for nome_do_arquivo in os.listdir(diretorio_dos_arquivos):

        # Filtra para processar apenas arquivos que terminam com .csv
        # if nome_do_arquivo.endswith('.csv'):
        if nome_do_arquivo.endswith('.csv') and nome_do_arquivo != arquivo_final_unificado:
            print('entrando')
            print(nome_do_arquivo)

            # Monta o caminho completo para o arquivo
            caminho_completo = os.path.join(diretorio_dos_arquivos, nome_do_arquivo)
            print(f"  -> Lendo o arquivo: {nome_do_arquivo}")

            try:
                # Lê o arquivo CSV, especificando o separador e o decimal
                df_temp = pd.read_csv(
                    caminho_completo,
                    sep=';',
                    decimal=','
                )

                # Adiciona o DataFrame recém-lido à nossa lista
                lista_de_dataframes.append(df_temp)

            except Exception as e:
                # Captura e exibe qualquer erro que possa ocorrer durante a leitura
                print(f"    AVISO: Ocorreu um erro ao ler o arquivo '{nome_do_arquivo}': {e}")
        else:
            print(f"  -> Ignorando (não é .csv): {nome_do_arquivo}")

    # 4. Consolidação e salvamento do resultado final
    if lista_de_dataframes:
        print("\n--- Unificando todos os dados lidos... ---")

        # Concatena (une) todos os DataFrames da lista em um único DataFrame
        df_unificado = pd.concat(lista_de_dataframes, ignore_index=True)

        # Salva o DataFrame consolidado no arquivo CSV final
        df_unificado.to_csv(
            os.path.join(os.path.join(os.getcwd(), diretorio_dos_arquivos),arquivo_final_unificado),
            index=False,
            sep=';',
            decimal=',',
            mode='w'
        )

        print(f"\n--- PROCESSO CONCLUÍDO ---")
        print(f"Os arquivos foram unificados com sucesso!")
        print(f"Resultados salvos em: '{arquivo_final_unificado}'")
        print(f"Total de linhas no arquivo final: {len(df_unificado)}")
    else:
        print("\n\n--- ATENÇÃO ---")
        print(f"Nenhum arquivo .csv foi encontrado ou processado no diretório '{diretorio_dos_arquivos}'.")
        print("O arquivo final unificado não foi gerado.")

### Verificação rotulacao do First e Last quantis
averiguar se as amostras pertencentes ao First e Last quantil foram corretamente rotuladas.

### Agregando todas os rotulos de todos os quantis e todos os sigmas em um unico dataframe
Já executado. Descomente se precisar executar novamente

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

# --- 1. Parâmetros ---
# Lista de quantis cujos arquivos de rótulos queremos unificar.
quantis = [2, 4, 8, 16]

# Lista para armazenar os DataFrames de cada arquivo lido.
lista_de_dataframes = []

print("Iniciando a unificação dos arquivos de rótulos pré-calculados...")

# --- 2. Loop para Ler os Arquivos Existentes ---
for qq in tqdm(quantis, desc="Lendo arquivos por quantil"):
    # Constrói o caminho para o arquivo de entrada esperado
    # Ex: 'C:\SeuProjeto\2_quantis\resultados_rotulos_2_quantis.csv'
    input_folder = os.path.join(os.getcwd(), f'{qq}_quantis')
    input_filename = f'resultados_rotulos_{qq}_quantis.csv' # Note que adicionei a extensão .csv
    input_filepath = os.path.join(input_folder, input_filename)

    # Verifica se o arquivo realmente existe antes de tentar lê-lo
    if os.path.exists(input_filepath):
        try:
            # Lê o arquivo CSV para um DataFrame temporário
            df_temp = pd.read_csv(
                input_filepath,
                sep=';',
                decimal=','
            )
            # Adiciona o DataFrame lido à nossa lista
            lista_de_dataframes.append(df_temp)
            print(f"\n  -> Arquivo para qq={qq} lido com sucesso. {len(df_temp)} linhas adicionadas.")
        except Exception as e:
            print(f"\n  -> ERRO ao ler o arquivo para qq={qq}: {e}")
    else:
        print(f"\n  -> AVISO: Arquivo não encontrado para qq={qq}, pulando: {input_filepath}")

# --- 3. Unificação e Salvamento Final ---

# Verifica se algum DataFrame foi carregado
if lista_de_dataframes:
    print("\nConcatenando todos os DataFrames carregados...")
    # Concatena todos os DataFrames da lista em um único DataFrame grande
    df_unificado = pd.concat(lista_de_dataframes, ignore_index=True)

    # Garante que a pasta de saída 'rotulos' exista
    output_folder = os.path.join(os.getcwd(), 'rotulos')
    os.makedirs(output_folder, exist_ok=True)

    # Define o caminho completo do arquivo de saída
    output_filename = os.path.join(output_folder, 'rotulos_unificados_todos_quantis_todos_sigmas.csv')

    print(f"Salvando arquivo CSV unificado em: {output_filename}")
    # Salva o DataFrame unificado em um novo arquivo CSV
    df_unificado.to_csv(
        output_filename,
        index=False,      # Não salva o índice do DataFrame como uma coluna
        sep=';',          # Usa ponto e vírgula como separador
        decimal=','       # Usa vírgula como separador decimal
    )

    print("\nProcesso concluído com sucesso!")
    print(f"O arquivo CSV final tem {len(df_unificado)} linhas e {len(df_unificado.columns)} colunas.")
else:
    print("\nNenhum arquivo de rótulos foi encontrado para processar. O arquivo final não foi gerado.")

## Carregamento de dados necessasrios ja calculados - Rotulos, First e Last quantis

## Graficos de linha ilustrando a taxa de concordância nos quantis das extremidades

In [ ]:
# # Lembrando que os rotulos originais, sem pertubacao, se referente àqueles onde o valor da pertubação é zero, em qualquer sigma e valor de quantil
# # Carregar o arquivo salvo
# output_folder = os.path.join(os.getcwd(), 'rotulos')
# output_filename = os.path.join(output_folder, 'rotulos_unificados_todos_quantis_todos_sigmas.csv')
# df_rotulos = pd.read_csv(output_filename, sep=';', decimal=',')
# df_rotulos.head()

In [ ]:
pwd

In [ ]:
# CARREGAMENTO DOS FIRSTS E LASTS QUANTIS
# --- 1. CONFIGURAÇÃO ---

# Lista de quantis para procurar as pastas e arquivos. A ordem aqui definirá a ordem nas listas de resultado.
quantis = [2, 4, 8, 16, 32, 64]

# Diretório raiz onde as pastas '{qq}_quantis' estão localizadas.
diretorio_raiz = os.getcwd()

# ALTERADO: Inicializa duas listas vazias.
first_quantis = []
last_quantis = []

print(f"Iniciando a busca e organização de arquivos em listas: {diretorio_raiz}")

# --- 2. LOOP PRINCIPAL PARA LER OS ARQUIVOS E ADICIONAR ÀS LISTAS ---

for qq in tqdm(quantis, desc="Processando quantis"):
    # Constrói o caminho para a pasta do quantil atual
    folder_path = os.path.join(diretorio_raiz, f'{qq}_quantis')

    # --- Processa o arquivo 'first_quantil' ---
    first_filepath = os.path.join(folder_path, f'first_quantil_{qq}.csv')

    if os.path.exists(first_filepath):
        try:
            # Lê o DataFrame
            df_first = pd.read_csv(first_filepath, sep=';', decimal=',')
            # ALTERADO: Adiciona o DataFrame à lista 'first_quantis'
            first_quantis.append(df_first)
        except Exception as e:
            print(f"\nOcorreu um erro ao ler o arquivo {first_filepath}: {e}")
            first_quantis.append(None) # Adiciona None em caso de erro de leitura
    else:
        print(f"\nAVISO: Arquivo 'first' não encontrado para qq={qq}, adicionando None à lista.")
        # ALTERADO: Adiciona None para manter o alinhamento dos índices
        first_quantis.append(None)

    # --- Processa o arquivo 'last_quantil' ---
    last_filepath = os.path.join(folder_path, f'last_quantil_{qq}.csv')

    if os.path.exists(last_filepath):
        try:
            df_last = pd.read_csv(last_filepath, sep=';', decimal=',')
            # ALTERADO: Adiciona o DataFrame à lista 'last_quantis'
            last_quantis.append(df_last)
        except Exception as e:
            print(f"\nOcorreu um erro ao ler o arquivo {last_filepath}: {e}")
            last_quantis.append(None)
    else:
        print(f"\nAVISO: Arquivo 'last' não encontrado para qq={qq}, adicionando None à lista.")
        # ALTERADO: Adiciona None para manter o alinhamento dos índices
        last_quantis.append(None)

# --- 3. VERIFICAÇÃO FINAL ---
print("\n--- Processo de Carregamento Concluído ---")

# ALTERADO: Verificação para a lista 'first_quantis'
print("\nResumo dos dados carregados na lista 'first_quantis':")
for i, df in enumerate(first_quantis):
    qq_correspondente = quantis[i]
    if df is not None:
        print(f"  - Posição {i} (qq={qq_correspondente}) carregada. Dimensões: {df.shape}")
    else:
        print(f"  - Posição {i} (qq={qq_correspondente}) não foi carregada (arquivo não encontrado ou erro).")

# ALTERADO: Verificação para a lista 'last_quantis'
print("\nResumo dos dados carregados na lista 'last_quantis':")
for i, df in enumerate(last_quantis):
    qq_correspondente = quantis[i]
    if df is not None:
        print(f"  - Posição {i} (qq={qq_correspondente}) carregada. Dimensões: {df.shape}")
    else:
        print(f"  - Posição {i} (qq={qq_correspondente}) não foi carregada (arquivo não encontrado ou erro).")